# Silver → Gold 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## fact_chamados

In [0]:
df_pesquisa_satisfacao = (
    spark.table(f"{silver_schema}.pesquisa_satisfacao")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_custo = (
    spark.table(f"{silver_schema}.custos")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_chamados = (
    spark.table(f"{silver_schema}.tb_chamados")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_fact = (
    df_chamados
    .join(df_pesquisa_satisfacao, on="id_chamado", how="left")
    .join(df_custo, on="id_chamado", how="left")
    .join(df_gold_motivos.withColumnRenamed("nome_motivo","motivo"), on="motivo", how="left")
    .join(df_gold_canais.withColumnRenamed("nome_canais","canal"), on="canal", how="left")
    .select(
        "id_chamado",
        F.col("id_cliente").cast("decimal(18,2)").alias("id_cliente"),
        F.col("id_atendente").cast("decimal(18,2)").alias("id_atendente"),
        F.col("id_motivo").cast("decimal(18,2)").alias("id_motivo"),
        F.col("id_canal").cast("decimal(18,2)").alias("id_canal"),
        F.col("id_data").cast("decimal(18,2)").alias("id_data"),
        F.col("resolvido").cast("decimal(18,2)").alias("resolvido"),
        F.col("custo").cast("decimal(18,2)").alias("custo"),
        F.col("nota_atendimento").cast("decimal(18,2)").alias("nota_atendimento"),
        F.col("tempo_espera").cast("decimal(18,2)").alias("tempo_espera_seg"),
        F.col("tempo_atendimento").cast("decimal(18,2)").alias("tempo_atendimento_seg")
    )
)

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_chamados")

print(f"fact_chamados: {df_fact.count()} chamados")
display(df_fact.limit(10))